# Tier 2 Assignment 3: Hi-C Analysis and Motif Discovery

**Covers**: Contact matrices, compartments, TADs, contact decay, PPM/PWM, information content, sequence scoring, motif enrichment

---

## Grading Rubric

| Problem | Points | Difficulty |
|---------|--------|------------|
| 1. Contact Matrix Normalization | 10 | (1 star) |
| 2. Contact Decay (P(s) Curve) | 15 | (2 stars) |
| 3. Insulation Score | 15 | (2 stars) |
| 4. A/B Compartment from Correlation | 10 | (3 stars) |
| 5. PPM and PWM Construction | 10 | (1 star) |
| 6. Information Content | 10 | (2 stars) |
| 7. PWM Sequence Scoring | 15 | (2 stars) |
| 8. Fisher's Exact Motif Enrichment | 15 | (3 stars) |
| **Total** | **100** | |

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
The methods here are applied in sequence analysis, annotation, motif/protein interpretation, and evidence building from biological databases.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Similarity is not identity: alignments, scores, and biological function are related but not equivalent.
- Database provenance matters: always track which release/version generated your result.
- Threshold choices (e-value, identity, score) strongly change sensitivity vs specificity.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


In [ ]:
import numpy as np
import math
from math import log2, comb, sqrt

---

## Part 1: Hi-C Analysis (Topic 14)

Hi-C is a genome-wide chromosome conformation capture technique. The raw output is
a **contact matrix** where entry `[i][j]` counts how often genomic bins `i` and `j`
were found in spatial proximity. Problems 1–4 implement core Hi-C analysis steps.

---

## Problem 1: Contact Matrix Normalization (1 star)

Raw Hi-C contact matrices are biased by differences in bin coverage (mappability,
GC content, restriction site density). A standard correction divides each entry by
the geometric mean of its row and column sum:

```
normalized[i][j] = raw[i][j] / sqrt(row_sum[i] * col_sum[j])
```

If `row_sum[i]` or `col_sum[j]` is zero, set `normalized[i][j] = 0.0`.

**Example**:
```
raw = [[4, 2], [2, 1]]
row_sums = [6, 3], col_sums = [6, 3]
normalized[0][0] = 4 / sqrt(6*6) = 4/6 ≈ 0.667
```

**Grading**: 10 points — 4 pts correct formula, 3 pts zero-division handling,
3 pts correct return type (list of lists of floats)

In [ ]:
def normalize_contact_matrix(matrix: list[list[int]]) -> list[list[float]]:
    """
    Coverage-normalize a square Hi-C contact matrix.

    normalized[i][j] = matrix[i][j] / sqrt(row_sum[i] * col_sum[j])
    Set result to 0.0 if row_sum[i] or col_sum[j] is zero.

    Args:
        matrix: Square contact matrix as list of lists of ints

    Returns:
        Normalized matrix as list of lists of floats
    """
    # YOUR CODE HERE
    pass


# Test
raw = [
    [4, 2, 0],
    [2, 6, 2],
    [0, 2, 4],
]
normed = normalize_contact_matrix(raw)
print("Normalized matrix:")
for row in normed:
    print([round(x, 4) for x in row])

# Sanity check: diagonal should be highest in each row after normalization
# Row sums should be equal for a well-normalized symmetric matrix

---

## Problem 2: Contact Decay (P(s) Curve) (2 stars)

The **contact probability P(s)** as a function of genomic distance `s` is a
fundamental Hi-C quality metric and reveals polymer properties of chromatin.
For each diagonal offset `d` (0 = main diagonal, 1 = one bin away, etc.),
average all entries `matrix[i][i+d]` for `i` in `0..n-1-d`.

Return a list of `(distance_bp, avg_contact)` tuples where
`distance_bp = d * bin_size`, sorted by distance ascending.

**Example** (5×5 matrix, bin_size=10000):
```
d=0: average of [m[0][0], m[1][1], m[2][2], m[3][3], m[4][4]]
d=1: average of [m[0][1], m[1][2], m[2][3], m[3][4]]
...
```

**Grading**: 15 points — 5 pts correct diagonal averaging, 5 pts correct distance
calculation, 5 pts correct sorting and return format

In [ ]:
def contact_decay(matrix: list[list[float]], bin_size: int) -> list[tuple]:
    """
    Compute average contact frequency at each genomic distance.

    For diagonal offset d, average matrix[i][i+d] for all valid i.
    Distance in bp = d * bin_size.

    Args:
        matrix:   Square contact matrix (list of lists)
        bin_size: Size of each bin in base pairs

    Returns:
        List of (distance_bp, avg_contact) tuples sorted by distance
    """
    # YOUR CODE HERE
    pass


# Test
test_matrix = [
    [10, 5, 2, 1, 0],
    [ 5, 8, 4, 2, 1],
    [ 2, 4, 9, 5, 2],
    [ 1, 2, 5, 7, 4],
    [ 0, 1, 2, 4, 6],
]
decay = contact_decay(test_matrix, bin_size=10_000)
print("Contact decay P(s):")
for dist, avg in decay:
    print(f"  d={dist:>8} bp: avg_contact = {avg:.4f}")

---

## Problem 3: Insulation Score (2 stars)

The **insulation score** measures how many contacts cross a given genomic position.
For bin `i` with window `w`, it is the mean of the square submatrix
`matrix[i-w : i, i : i+w]` — i.e., the `w×w` block of contacts between the
`w` bins upstream and `w` bins downstream of `i`.

TAD boundaries appear as **local minima** of the insulation score (few contacts
cross the boundary). Bins within `w` of either edge get score `0.0`.

Return a list of floats (one per bin).

**Example** (window=1 for a 4×4 matrix):
```
insulation[0] = 0.0  (within w=1 of left edge)
insulation[1] = mean(matrix[0:1, 1:2]) = matrix[0][1]
insulation[2] = mean(matrix[1:2, 2:3]) = matrix[1][2]
insulation[3] = 0.0  (within w=1 of right edge)
```

**Grading**: 15 points — 5 pts correct submatrix indexing, 5 pts edge handling,
5 pts mean computation

In [ ]:
def insulation_score(matrix: list[list[float]], window: int) -> list[float]:
    """
    Compute insulation score for each bin in a contact matrix.

    For bin i: mean of matrix[i-w:i, i:i+w] (the w×w upstream-downstream block).
    Bins with i < window or i > n - window - 1 get score 0.0.

    Args:
        matrix: Square contact matrix (list of lists)
        window: Half-window size in bins

    Returns:
        List of float insulation scores, one per bin
    """
    # YOUR CODE HERE
    pass


# Test
# Two-TAD synthetic matrix: high intra-TAD contacts, low inter-TAD
tad_matrix = [
    [10,  8,  1,  1,  0],
    [ 8, 10,  1,  0,  1],
    [ 1,  1, 10,  8,  1],
    [ 1,  0,  8, 10,  1],
    [ 0,  1,  1,  1, 10],
]
scores = insulation_score(tad_matrix, window=1)
print("Insulation scores:", [round(s, 4) for s in scores])
print("TAD boundary expected at bin with lowest non-zero insulation score")

---

## Problem 4: A/B Compartment from Correlation (3 stars)

A/B compartments represent active (A) and inactive (B) chromatin. The standard
method uses PCA on the correlation matrix of a normalized Hi-C matrix:

1. Compute the Pearson correlation matrix `C` between each pair of rows
   of the normalized contact matrix (use `numpy`)
2. Center `C`: `M = C - C.mean(axis=0)`
3. Find the first eigenvector of `M @ M.T` using **power iteration**
   (50 iterations is sufficient): start with a random unit vector, repeatedly
   multiply by `M @ M.T` and re-normalize
4. Assign compartment: `+1` (A) if eigenvector component > 0, `-1` (B) otherwise

Return a list of `1` or `-1` values, one per bin.
Use `numpy.random.seed(42)` at the start for reproducibility.

**Grading**: 10 points — 3 pts correlation matrix, 3 pts power iteration,
4 pts correct sign assignment and return

In [ ]:
def ab_compartments(norm_matrix: list[list[float]]) -> list[int]:
    """
    Assign A/B compartments via correlation PCA (power iteration).

    Steps:
      1. Compute Pearson correlation matrix C between rows of norm_matrix.
      2. Center C: M = C - C.mean(axis=0).
      3. Compute first eigenvector of M @ M.T via 50 power iterations.
      4. Return [1 if v > 0 else -1 for v in eigenvector].

    Args:
        norm_matrix: Normalized square contact matrix (list of lists)

    Returns:
        List of 1 (compartment A) or -1 (compartment B) per bin
    """
    np.random.seed(42)
    # YOUR CODE HERE
    pass


# Test
# Synthetic matrix: bins 0-2 form compartment A (correlated),
# bins 3-5 form compartment B (anti-correlated with A)
import random
random.seed(0)
# Block structure: strong intra-block, weak inter-block
ab_mat = [
    [10, 8, 7, 1, 1, 0],
    [ 8,10, 8, 1, 0, 1],
    [ 7, 8,10, 0, 1, 1],
    [ 1, 1, 0,10, 8, 7],
    [ 1, 0, 1, 8,10, 8],
    [ 0, 1, 1, 7, 8,10],
]
compartments = ab_compartments(ab_mat)
print("Compartment assignments:", compartments)
print("Expected: first 3 bins same label, last 3 bins opposite label")

---

## Part 2: Motif Discovery (Topic 15)

Transcription factors bind DNA at specific sequence motifs. Problems 5–8 build
the core toolkit for representing, scoring, and testing motifs statistically.

---

## Problem 5: PPM and PWM Construction (1 star)

Given a list of aligned DNA sequences, build:

- **PPM** (Position Probability Matrix): frequency of each base at each position,
  with pseudocount 0.1 added to each count before dividing by the total.
  `ppm[base][pos]` where base ∈ `'ACGT'`.

- **PWM** (Position Weight Matrix): log-odds score relative to uniform background.
  `pwm[base][pos] = log2(ppm[base][pos] / 0.25)`

**Example** (3 sequences of length 2, position 0 has A,A,G):
```
counts_A[0] = 2 + 0.1 = 2.1
counts_G[0] = 1 + 0.1 = 1.1
counts_C[0] = 0 + 0.1 = 0.1, counts_T[0] = 0 + 0.1 = 0.1
total = 3 + 4*0.1 = 3.4
ppm['A'][0] = 2.1 / 3.4 ≈ 0.6176
```

**Grading**: 10 points — 4 pts correct PPM with pseudocounts,
3 pts correct PWM formula, 3 pts correct dict structure

In [ ]:
def build_ppm_pwm(sequences: list[str]) -> tuple[dict, dict]:
    """
    Build PPM and PWM from a list of aligned DNA sequences.

    PPM: ppm[base][pos] = (count + 0.1) / (n_seqs + 4*0.1)
    PWM: pwm[base][pos] = log2(ppm[base][pos] / 0.25)
    Both dicts have keys in 'ACGT' and values are lists indexed by position.

    Args:
        sequences: List of DNA strings, all the same length

    Returns:
        (ppm, pwm) where each is a dict with keys 'A','C','G','T'
        and values are lists of floats (one per position)
    """
    # YOUR CODE HERE
    pass


# Test
motif_seqs = [
    "ACGT",
    "ACGT",
    "ACGT",
    "ACGT",
    "ACGA",
]
ppm, pwm = build_ppm_pwm(motif_seqs)
print("PPM:")
for base in "ACGT":
    print(f"  {base}: {[round(v, 4) for v in ppm[base]]}")
print("PWM:")
for base in "ACGT":
    print(f"  {base}: {[round(v, 4) for v in pwm[base]]}")

---

## Problem 6: Information Content (2 stars)

Information content (IC) quantifies how specific a motif position is.
A perfectly conserved position has IC = 2 bits; a random position has IC = 0 bits.

For each position `p`:
```
IC[p] = 2 + sum_{b in ACGT} ppm[b][p] * log2(ppm[b][p])
```
Terms where `ppm[b][p] = 0` contribute 0 to the sum (not `-inf`).

Also compute total IC = sum of all position ICs.

Return `(ic_per_position, total_ic)` where `ic_per_position` is a list of floats.

**Grading**: 10 points — 4 pts correct per-position formula, 3 pts zero handling,
3 pts total IC

In [ ]:
def information_content(ppm: dict) -> tuple[list[float], float]:
    """
    Compute information content per position and total IC from a PPM.

    IC[p] = 2 + sum_b( ppm[b][p] * log2(ppm[b][p]) )
    (terms with ppm[b][p] == 0 contribute 0, not -inf)

    Args:
        ppm: Position probability matrix, ppm[base][pos] for base in 'ACGT'

    Returns:
        (ic_per_position, total_ic)
        ic_per_position: list of floats, one per position
        total_ic: sum of all ic_per_position values
    """
    # YOUR CODE HERE
    pass


# Test using the PPM from Problem 5
ic_per_pos, total_ic = information_content(ppm)
print("IC per position:", [round(v, 4) for v in ic_per_pos])
print(f"Total IC: {total_ic:.4f} bits")
print("Expected: positions 0-2 near 2.0 bits (conserved), position 3 lower")

---

## Problem 7: PWM Sequence Scoring (2 stars)

To find where a motif occurs in a sequence, score every possible window using
the PWM. The score for a window starting at position `i` is:

```
score(i) = sum_{j=0}^{L-1} pwm[seq[i+j]][j]
```

where `L` is the motif length (width of the PWM).

Scan **both** the forward sequence and its reverse complement. Return the
overall best hit:

```python
{"best_score": float, "best_position": int, "strand": "+" or "-"}
```

`best_position` is 0-based on the **original** sequence. For the reverse
complement, the position of a window starting at offset `k` in the RC
corresponds to `len(seq) - L - k` on the original strand.

Reverse complement: A↔T, C↔G; reverse the string.

**Grading**: 15 points — 5 pts forward strand scoring, 5 pts reverse complement
scoring and position conversion, 5 pts correct return dict

In [ ]:
def score_sequence(pwm: dict, sequence: str) -> dict:
    """
    Find the best PWM match in a sequence (both strands).

    Score a window of length L starting at position i:
      score = sum(pwm[seq[i+j]][j] for j in range(L))
    Scan all valid windows on both forward and reverse complement strands.

    Args:
        pwm:      Position weight matrix, pwm[base][pos] for base in 'ACGT'
        sequence: DNA sequence string to scan

    Returns:
        dict with keys:
            best_score (float): highest window score found
            best_position (int): 0-based position on the original sequence
            strand (str): '+' or '-'
    """
    # YOUR CODE HERE
    pass


# Test
# Build a PWM for the motif ACGT
pwm_seqs = ["ACGT"] * 10
_, test_pwm = build_ppm_pwm(pwm_seqs)

# Embed the motif at position 3 of a longer sequence
test_seq = "NNNACGTNNN"
result = score_sequence(test_pwm, test_seq)
print("Best hit:", result)
print("Expected: best_position=3, strand='+'")

# Also test reverse complement detection
rc_seq = "NNNACGTNNN"  # reverse complement of ACGT is ACGT (palindrome)
rc_result = score_sequence(test_pwm, rc_seq)
print("RC test:", rc_result)

---

## Problem 8: Fisher's Exact Motif Enrichment (3 stars)

To test whether a motif is significantly enriched in a set of peaks (foreground)
versus background sequences, use Fisher's exact test.

**Steps**:
1. Count hits in foreground: sequences where `max(score_sequence PWM score) >= threshold`.
   Call this `a`. Foreground non-hits: `b = fg_total - a`.
2. Count hits in background: `c`. Background non-hits: `d = bg_total - c`.
3. Odds ratio: `(a/b) / (c/d)` = `(a*d) / (b*c)`. Use 0.0 if denominator is 0.
4. Fisher p-value (one-tailed, more extreme = more enriched in fg):
   Sum hypergeometric probabilities for all tables with fg_hits ≥ a.

   The probability of a specific 2×2 table with fg_hits=`k` is:
   ```
   P(k) = C(a+b, k) * C(c+d, a+c-k) / C(n, a+c)
   ```
   where `n = a+b+c+d` and `a+c` = total hits across both groups.
   Sum over all `k` from `a` to `min(a+b, a+c)`.

   Use `math.comb` for binomial coefficients.

Return `{"odds_ratio": float, "p_value": float, "fg_hits": int, "bg_hits": int}`.

**Grading**: 15 points — 4 pts hit counting, 4 pts odds ratio, 7 pts Fisher p-value
using hypergeometric formula

In [ ]:
def motif_enrichment(
    foreground: list[str],
    background: list[str],
    pwm: dict,
    threshold: float,
) -> dict:
    """
    Test motif enrichment in foreground vs background using Fisher's exact test.

    A sequence is a 'hit' if its best PWM score (either strand) >= threshold.
    Fisher p-value: sum of hypergeometric probabilities for tables as extreme
    or more extreme (fg_hits >= observed).

    P(k) = C(fg_total, k) * C(bg_total, total_hits - k) / C(n, total_hits)
    p_value = sum_{k=fg_hits}^{min(fg_total, total_hits)} P(k)

    Args:
        foreground: List of foreground (peak) sequences
        background: List of background sequences
        pwm:        Position weight matrix dict
        threshold:  Minimum score to count a sequence as a motif hit

    Returns:
        dict with keys: odds_ratio (float), p_value (float),
                        fg_hits (int), bg_hits (int)
    """
    # YOUR CODE HERE
    pass


# Test
# Foreground: mostly contain ACGT motif
fg_seqs = [
    "TTTACGTAAA",
    "AAACGTTTT",
    "GGACGTCC",
    "TTACGTTT",
    "CCACGTGG",
    "AAATTTGGG",  # no motif
]
# Background: mostly random
bg_seqs = [
    "AAATTTGGG",
    "CCCGGGTTT",
    "TTTGGGCCC",
    "GGGAAACCC",
    "TTTACGTAA",  # one hit
    "AAACCCGGG",
    "CCCAAATTT",
    "TTTCCCAAA",
]

# Build a tight ACGT PWM
tight_pwm_seqs = ["ACGT"] * 20
_, tight_pwm = build_ppm_pwm(tight_pwm_seqs)

result = motif_enrichment(fg_seqs, bg_seqs, tight_pwm, threshold=3.0)
print("Enrichment result:", result)
print("Expected: fg_hits > bg_hits relative to group sizes, p_value < 0.1")

---

## Summary

In this assignment you implemented core algorithms for two major genomics domains:

**Hi-C Analysis (Topic 14)**:
1. **Contact Matrix Normalization**: correcting coverage bias via geometric-mean scaling
2. **Contact Decay P(s)**: measuring how contact frequency falls off with distance
3. **Insulation Score**: identifying TAD boundaries as minima of cross-boundary contact
4. **A/B Compartments**: separating active and inactive chromatin via correlation PCA

**Motif Discovery (Topic 15)**:
5. **PPM/PWM Construction**: summarizing aligned binding sites as probability and weight matrices
6. **Information Content**: quantifying motif conservation in bits
7. **PWM Sequence Scoring**: scanning both strands to locate the best motif match
8. **Fisher's Exact Test**: testing motif enrichment using the hypergeometric distribution